# Public Transit Honor System vs. Fare Gates

Michael Tian, Michael Mehall, Evan Crow, Leandros Manwaring, Ian Solberg 

April 2026

---

In [5]:
import pandas as pd
import numpy as np
from Research_Framework.ResearchHandler import ResearchHandler
import Research_Framework.transforms as tf
from mbta_handling import blank, build_policy_flow 

---

#### Setup Cell, do not execute - files too large for GH storage.

In [ ]:
df = build_policy_flow(nodes_fp="data/mbta_rapid_transit/MBTA_NODE.shp", gse_fp="data/GSE/GSE.csv", travel_times_fp="data/TravelTimes_2026/2026-02_HRTravelTimes.csv")
mbta = ResearchHandler(df, handling_function=blank, initialize=False)
# mbta.data.to_csv("data/output_data/mbta_gate_flows_clean.csv", index=False)

---

## Real Setup:

In [6]:
fp = "data/output_data/mbta_gate_flows_clean.csv"
mbta = ResearchHandler(fp, handling_function=blank, initialize=True)
mbta.data

,station_line_source,station_line_destination,line,time_bucket,avg_travel_time_sec,source_avg_gated_entries,dest_avg_gated_entries
0,"Kendall/MIT, RED","Wollaston, RED",RED,PM Rush,1527.120163,3996.478442,212.271115
1,"Kendall/MIT, RED","Park Street, RED",RED,PM Rush,227.785782,3996.478442,1481.583148
2,"Kendall/MIT, RED","Savin Hill, RED",RED,PM Rush,1072.838900,3996.478442,238.299365
3,"Kendall/MIT, RED","Braintree, RED",RED,PM Rush,2253.193416,3996.478442,359.482714
4,"Kendall/MIT, RED","Quincy Center, RED",RED,PM Rush,1712.395538,3996.478442,525.383793
...,...,...,...,...,...,...,...
5553,"State, ORANGE","Wellington, ORANGE",ORANGE,Early Morning,642.332418,0.000000,421.001378
5554,"State, ORANGE","Wellington, ORANGE",ORANGE,Evening,635.147175,0.000000,183.455115
5555,"State, ORANGE","Wellington, ORANGE",ORANGE,Late Night,592.536170,0.000000,85.692929
5556,"State, ORANGE","Wellington, ORANGE",ORANGE,Midday,684.000000,0.000000,843.853244


# MBTA Policy Flow Dataset

**5,558 rows** — each row is a directed station pair on a rapid transit line during a time-of-day bucket, with average travel time and fare gate activity for February 2026.

## Columns

| Column | Description |
|--------|-------------|
| `station_line_source` | Origin station and line (e.g. `Kendall/MIT, RED`) |
| `station_line_destination` | Destination station and line |
| `line` | Rapid transit line: RED, ORANGE, or BLUE |
| `time_bucket` | Period of day: Early Morning, AM Rush, Midday, PM Rush, Evening, Late Night |
| `avg_travel_time_sec` | Mean travel time in seconds across all February 2026 trips |
| `source_avg_gated_entries` | Mean fare gate entries at origin station during that time bucket |
| `dest_avg_gated_entries` | Mean fare gate entries at destination station during that time bucket |

## Time Buckets

| Bucket | Hours |
|--------|-------|
| Early Morning | 5:00–7:00 |
| AM Rush | 7:00–10:00 |
| Midday | 10:00–15:00 |
| PM Rush | 15:00–19:00 |
| Evening | 19:00–22:00 |
| Late Night | 22:00–5:00 |

## Notes

- Gate entries of `0` indicate stations with no fare gate data in the source (GSE) dataset (State, Massachusetts Avenue on Orange; State on Blue).
- Gate entry values are summed across 30-minute periods within each time bucket, after averaging across service dates.
- Source data: MBTA Node shapefile, Gated Station Entries (GSE), and High-Resolution Travel Times.

---

# Analysis

In [7]:
mbta.data["associated_gate_total"] = mbta.data["source_avg_gated_entries"] + mbta.data["dest_avg_gated_entries"]  

mbta.set_dependent("associated_gate_total")

Dependent variable set to: associated_gate_total


In [ ]:
mbta.add_independents()
mbta.add_controls()

# TODO implement analysis